## Project Summary

This notebook investigates one of the most fascinating questions in theoretical astrophysics:

**Can we distinguish a traversable wormhole from a black hole using gravitational lensing?**

Although no wormhole has ever been observed, General Relativity predicts that both black holes and wormholes bend light travelling through spacetime. However, because a traversable wormhole connects two different regions of spacetime instead of ending in an event horizon, its lensing behaviour is expected to differ from that of a black hole.

In this project we computationally simulate both systems, compare their gravitational lensing signatures, and propose observational criteria that future telescopes may use to identify potential wormhole candidates.

---

## Objectives

• Build a Schwarzschild black hole lensing simulation.

• Build a Morris–Thorne traversable wormhole lensing simulation.

• Perform ray tracing for thousands of light rays.

• Visualize photon trajectories.

• Generate simulated lensing images.

• Compare black hole and wormhole lensing.

• Measure quantitative differences such as:
    - Einstein ring radius
    - Image multiplicity
    - Magnification
    - Central brightness
    - Photon travel time

• Develop a set of observational detection criteria capable of distinguishing wormholes from classical black holes.

---

## Expected Outcome

By the end of this notebook we will have a complete computational framework capable of producing side-by-side gravitational lensing simulations and identifying observable signatures that may distinguish traversable wormholes from black holes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

plt.style.use("dark_background")

##  Define Physical Constants


General Relativity describes spacetime using physical constants such as the gravitational constant and the speed of light.

Rather than repeatedly typing these values throughout the notebook, we define them once here for consistency and readability.

In [ ]:
# Physical constants (SI Units)

G = 6.67430e-11
c = 299792458

M_sun = 1.98847e30

## Define Compact Object Parameters

In this section we specify the properties of the compact object that will produce gravitational lensing.

Initially we simulate a Schwarzschild black hole with a chosen mass. Later, this object will be replaced by a traversable wormhole while keeping the remaining simulation identical.

This allows a fair comparison between the two spacetime geometries.

## Compute the Schwarzschild Radius
The Schwarzschild radius defines the event horizon of a non-rotating black hole.

It represents the radius at which the escape velocity equals the speed of light.

Photons crossing this boundary cannot escape, producing the familiar black hole shadow observed by the Event Horizon Telescope.

In [ ]:
M = 4e6 * M_sun

Rs = (2 * G * M) / c**2

print(Rs)

The calculated Schwarzschild radius is

$$
R_s \approx 1.18 \times 10^{10}\ \mathrm{m}
$$

which is approximately **11.8 million kilometres**.

This is the radius of the event horizon for a black hole with a mass of

$$
M = 4 \times 10^6\,M_\odot,
$$

where \(M_\odot\) denotes the mass of the Sun. This is comparable to the mass of the supermassive black hole at the centre of the Milky Way.

Any photon or object crossing the event horizon (\(R_s\)) cannot escape the black hole's gravitational pull. In the following simulations, this radius serves as the characteristic length scale for modelling photon trajectories, spacetime curvature, and gravitational lensing.

## Build the Lens Plane


Before we study gravitational lensing, we construct a two-dimensional image plane representing the observer's view of the sky.

Each point in this plane corresponds to one incoming light ray.

The simulation will later trace every ray backwards through curved spacetime to determine its apparent position.
---



## Simulating Photon Deflection


In General Relativity, gravity curves spacetime, causing light rays to follow curved paths instead of travelling in perfectly straight lines.

For a Schwarzschild (non-rotating) black hole, the weak-field approximation predicts that the bending angle depends on the impact parameter of the photon.

The closer a photon passes to the black hole, the stronger the deflection.

In this section, we calculate the deflection angle for thousands of incoming light rays using Einstein's weak-field approximation.

Although this approximation is not accurate near the event horizon, it provides an starting point before implementing full relativistic ray tracing later in the notebook.


In [ ]:
# Number of simulated light rays
num_rays = 2000

# Impact parameters (meters)
# Rays passing closer to the black hole experience stronger bending
b = np.linspace(1.2 * Rs, 30 * Rs, num_rays)

# Einstein weak-field deflection angle
alpha = (4 * G * M) / (c**2 * b)

print(f"Number of rays simulated : {num_rays}")
print(f"Minimum impact parameter : {b.min():.2e} m")
print(f"Maximum impact parameter : {b.max():.2e} m")

The simulation generated **2,000** incoming light rays with impact parameters ranging from

$$
1.42 \times 10^{10}\ \mathrm{m}
$$

to

$$
3.54 \times 10^{11}\ \mathrm{m}.
$$

The minimum impact parameter is slightly larger than the Schwarzschild radius, allowing photons to pass very close to the black hole, where gravitational lensing is strongest. The maximum impact parameter corresponds to photons travelling much farther from the black hole, where the gravitational deflection is comparatively weak.

This range of impact parameters enables the simulation to capture both **strong-field** and **weak-field** gravitational lensing, providing a representative sample of photon trajectories for the subsequent lensing analysis.

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(b / Rs, alpha, color="cyan", linewidth=2)

plt.xlabel("Impact Parameter (Rs)")
plt.ylabel("Deflection Angle (radians)")
plt.title("Photon Deflection Around a Schwarzschild Black Hole")

plt.grid(alpha=0.3)

plt.show()


The graph shows how the gravitational deflection angle changes with the photon's impact parameter.

The downward-curving shape indicates an inverse relationship between these two quantities.

Photons passing very close to the black hole (small impact parameter) experience the strongest gravitational bending. As the distance from the black hole increases, the deflection angle decreases rapidly.

The curve gradually flattens at larger impact parameters, showing that photons travelling far from the black hole are only weakly affected by spacetime curvature.

This behaviour is consistent with Einstein's prediction that gravitational lensing becomes strongest near massive compact objects and weakens with increasing distance.

## Ray Tracing the Background Sky

The previous section calculated how much each light ray bends.

In this section, we use those bending angles to determine where each photon appears in the observer's image plane.

Ray tracing is one of the most important techniques in computational astrophysics. Instead of tracing photons from the source to the observer, we work backwards by tracing rays from the observer through curved spacetime.

This method allows us to reconstruct the distorted appearance of distant galaxies caused by gravitational lensing.

Although this implementation uses a simplified weak-field approximation, the overall workflow is similar to that used in professional ray-tracing simulations.

In [ ]:
np.random.seed(42)

num_stars = 1200

# Random positions between -1 and 1
x = np.random.uniform(-1, 1, num_stars)
y = np.random.uniform(-1, 1, num_stars)

plt.figure(figsize=(8,8))

plt.scatter(x, y,
            s=2,
            color="white")

plt.title("Background Star Field")

plt.xlim(-1,1)
plt.ylim(-1,1)

plt.gca().set_facecolor("black")

plt.show()

Before applying gravitational lensing, we first generate a simple background star field.

Each white point represents a distant star.

In reality these stars would be located at enormous cosmological distances, but for visualization purposes they are randomly distributed across the image plane.

This image serves as the unlensed reference that will later be distorted by the gravitational field of the compact object.

## Compute Distance from Centre

In [ ]:
# Distance from image center

r = np.sqrt(x**2 + y**2)

# Avoid division by zero
r = np.maximum(r, 0.02)

Gravitational lensing depends on how close each light ray passes to the compact object.

The variable r represents the radial distance of every photon from the centre of the image.

Very small distances are replaced with a minimum value to prevent division-by-zero errors during later calculations.

## Apply Lensing

In [ ]:
# Simple lensing strength

k = 0.06

factor = 1 + k / r

x_lensed = x * factor
y_lensed = y * factor

This simplified lensing model stretches light rays outward depending on their distance from the centre.

Stars close to the compact object experience much stronger distortion than stars farther away.

Although this is not yet a full relativistic ray tracer, it captures the qualitative behaviour expected from gravitational lensing.

## Visualize Lensed Stars

In [ ]:
plt.figure(figsize=(8,8))

plt.scatter(
    x_lensed,
    y_lensed,
    s=2,
    color="cyan"
)

plt.title("Simplified Gravitational Lensing")

plt.xlim(-1.5,1.5)
plt.ylim(-1.5,1.5)

plt.gca().set_facecolor("black")

plt.show()


The gravitational field bends the paths of photons, changing their apparent positions.

Stars near the centre are displaced more strongly, while distant stars remain nearly unchanged.

This distortion is the essence of gravitational lensing and provides the foundation for more realistic black hole imaging.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,7))

# Original sky
ax[0].scatter(x, y,
              s=2,
              color="white")

ax[0].set_title("Original Sky")
ax[0].set_facecolor("black")
ax[0].set_xlim(-1,1)
ax[0].set_ylim(-1,1)

# Lensed sky
ax[1].scatter(x_lensed,
              y_lensed,
              s=2,
              color="cyan")

ax[1].set_title("After Gravitational Lensing")
ax[1].set_facecolor("black")
ax[1].set_xlim(-1.5,1.5)
ax[1].set_ylim(-1.5,1.5)

plt.show()

In this section, we implemented a simplified ray-tracing simulation to illustrate the effects of gravitational lensing.

A synthetic background star field was generated, and each star's apparent position was modified according to a basic lensing model. The resulting image demonstrates how gravity can distort the appearance of distant objects.

While this approach captures the qualitative behaviour of gravitational lensing, it remains an approximation. In the next section, we will refine the simulation by modelling the black hole shadow and photon ring, bringing the rendered image closer to observations from instruments such as the Event Horizon Telescope.

This section is intentionally a stepping stone. The next upgrade will implement a more realistic black hole image by adding the event horizon shadow, photon ring, and Einstein ring, before we later replace the simplified lensing with full Schwarzschild geodesic ray tracing using numerical integration.

## Simulating the Black Hole Shadow and Photon Ring



In the previous section, we observed how gravity bends the paths of distant light rays.

However, a real black hole produces additional observable features beyond simple lensing.

The most striking is the **black hole shadow**, a dark central region where photons are captured by the event horizon. Surrounding the shadow is the **photon ring**, a thin, bright ring formed by photons that orbit the black hole multiple times before escaping toward the observer.

These structures have been observed around supermassive black holes by the Event Horizon Telescope and represent one of the strongest confirmations of General Relativity.

In this section, we extend our simulation by adding a black hole shadow and an approximate photon ring to create a more realistic visualization of gravitational lensing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Image resolution
N = 700

x = np.linspace(-2, 2, N)
y = np.linspace(-2, 2, N)

X, Y = np.meshgrid(x, y)

R = np.sqrt(X**2 + Y**2)

# Background brightness decreases slightly with radius
image = np.exp(-0.25 * R)

## Create the Event Horizon Shadow

In [ ]:
# Schwarzschild shadow radius (scaled)
shadow_radius = 0.45

shadow = R < shadow_radius

image_shadow = image.copy()

image_shadow[shadow] = 0

The event horizon captures any photon that crosses its boundary.

To represent this effect, every pixel inside the chosen shadow radius is assigned zero brightness.

The result is the characteristic dark region at the centre of the black hole image.

## Create the Photon Ring

In [ ]:
ring_radius = 0.55
ring_width = 0.03

ring = np.abs(R - ring_radius) < ring_width

image_shadow[ring] = 2.5


Photons passing extremely close to the black hole may orbit it several times before escaping.

This concentration of photons produces a thin bright ring surrounding the shadow.

Although simplified, this ring captures the qualitative appearance predicted by General Relativity.

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    image_shadow,
    cmap="inferno",
    extent=(-2,2,-2,2),
    origin="lower"
)

plt.title("Simulated Schwarzschild Black Hole")

plt.xlabel("x")

plt.ylabel("y")

plt.colorbar(label="Brightness")

plt.show()

The simulated image displays three important regions:

- The bright background represents distant light sources.

- The dark central shadow corresponds to photons captured by the event horizon.

- The bright circular ring represents photons orbiting near the photon sphere before escaping toward the observer.

These features are consistent with the qualitative predictions of General Relativity for a Schwarzschild black hole

## Add Einstein Ring Enhancement

In [ ]:
einstein_radius = 1.1
einstein_width = 0.05

einstein_ring = np.abs(R - einstein_radius) < einstein_width

image_shadow[einstein_ring] += 0.8

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    image_shadow,
    cmap="inferno",
    extent=(-2,2,-2,2),
    origin="lower"
)

plt.title("Black Hole with Shadow, Photon Ring and Einstein Ring")

plt.colorbar(label="Brightness")

plt.show()

An additional Einstein ring has been included to illustrate the gravitational lensing of background light.

This ring forms when photons from a distant source are bent symmetrically around the compact object.

The complete image now contains:

- Background emission
- Event horizon shadow
- Photon ring
- Einstein ring

Together, these structures produce a simplified representation of the gravitational appearance of a Schwarzschild black hole.

In this section, we enhanced the gravitational lensing simulation by incorporating the defining visual features of a Schwarzschild black hole.

The event horizon shadow was modelled as a central dark region where photons are absorbed. A bright photon ring was added to represent light orbiting near the photon sphere, while an Einstein ring illustrated the lensing of distant background sources.

Although simplified, this model captures the key observational characteristics expected from a non-rotating black hole and provides a reference for comparison with traversable wormholes.

The next section will introduce the Morris–Thorne wormhole geometry. Unlike a black hole, a traversable wormhole does not possess an event horizon, allowing photons to pass through its throat. This fundamental difference is expected to produce distinctive gravitational lensing signatures that may serve as observable indicators of wormhole spacetimes.

Note: The following is a phenomenological (visual) model, not yet a full numerical solution of the Morris–Thorne metric. It is intended to demonstrate the expected qualitative appearance. Later, we can upgrade it to solve photon geodesics directly from the wormhole metric.

## Simulating a Traversable Wormhole


Unlike a Schwarzschild black hole, a traversable wormhole does not contain an event horizon.

Instead, spacetime is connected through a throat that links two distant regions of the universe. Photons approaching the throat are not necessarily absorbed; instead, they may pass through and emerge elsewhere.

As a result, the gravitational lensing produced by a wormhole is expected to differ from that of a black hole.

The main expected observational features include:

- No completely dark central shadow
- Bright throat region
- Multiple Einstein rings
- Additional photon paths
- Enhanced central luminosity

In this section, we construct a simplified visualization of these predicted features.

In [ ]:
# Start with the original background
wormhole_image = image.copy()

To ensure a fair comparison, the wormhole simulation begins with exactly the same background used for the black hole.

This allows any visual differences to arise from the spacetime geometry rather than the background itself.

## Create the Wormhole Throat

In [ ]:
# Wormhole throat radius
throat_radius = 0.45

# Bright throat
throat = R < throat_radius

wormhole_image[throat] = 2.8

Instead of creating a dark shadow, we model the wormhole throat as a bright central region.

This represents photons travelling through the wormhole and reaching the observer from the opposite side of spacetime.

Unlike a black hole, the centre is no longer completely dark.

## Primary Ring

In [ ]:
primary_radius = 0.55
primary_width = 0.03

primary_ring = np.abs(R - primary_radius) < primary_width

wormhole_image[primary_ring] = 3.2


The primary ring is analogous to the bright photon ring seen around black holes.

However, because photons can pass through the wormhole throat, the brightness distribution may differ from the Schwarzschild case.

## Secondary Einstein Ring

In [ ]:
secondary_radius = 0.95
secondary_width = 0.04

secondary_ring = np.abs(R - secondary_radius) < secondary_width

wormhole_image[secondary_ring] += 1.2


One possible observational signature of a traversable wormhole is the appearance of additional Einstein rings.

These extra rings arise because photons may follow multiple trajectories through curved spacetime before reaching the observer.

Although simplified, this additional ring illustrates the concept of image multiplicity predicted in some wormhole models.

## Outer Ring

In [ ]:
outer_radius = 1.35
outer_width = 0.05

outer_ring = np.abs(R - outer_radius) < outer_width

wormhole_image[outer_ring] += 0.8


A third ring is added to represent higher-order photon trajectories.

These rings are expected to be much fainter in reality, but including them helps illustrate one of the key qualitative differences between wormholes and classical black holes.

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    wormhole_image,
    cmap="inferno",
    origin="lower",
    extent=(-2,2,-2,2)
)

plt.title("Simulated Traversable Wormhole")

plt.xlabel("x")
plt.ylabel("y")

plt.colorbar(label="Brightness")

plt.show()


Unlike the black hole simulation, the wormhole image does not contain a completely dark central shadow.

Instead, the centre appears bright due to photons emerging from the wormhole throat.

Multiple concentric rings surround the throat, representing different photon trajectories predicted by theoretical wormhole models.

These visual differences provide candidate observational signatures that could distinguish traversable wormholes from black holes.


In this section, we developed a simplified visualization of a traversable wormhole.

The simulation differs from the Schwarzschild black hole in several important ways:

- The central region remains luminous rather than completely dark.
- Multiple concentric rings are present.
- Additional photon paths are represented through higher-order rings.

These features form the basis for the comparative analysis that follows. In the next section, the black hole and wormhole simulations will be displayed side by side, allowing their gravitational lensing signatures to be examined and compared directly.

##  Comparative Analysis of Black Hole and Wormhole Lensing

Having simulated both a Schwarzschild black hole and a traversable wormhole, we now compare their predicted observational signatures.

Rather than relying on visual inspection alone, we calculate quantitative metrics that can be measured directly from the simulated images.

The comparison focuses on several observable properties:

- Central brightness
- Ring brightness
- Radial brightness distribution
- Number of visible rings

These measurements provide an objective framework for distinguishing wormholes from black holes in future astronomical observations.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,7))

# Black Hole
ax[0].imshow(
    image_shadow,
    cmap="inferno",
    origin="lower",
    extent=(-2,2,-2,2)
)

ax[0].set_title("Schwarzschild Black Hole")

# Wormhole
ax[1].imshow(
    wormhole_image,
    cmap="inferno",
    origin="lower",
    extent=(-2,2,-2,2)
)

ax[1].set_title("Traversable Wormhole")

plt.tight_layout()
plt.show()


The two simulated compact objects are displayed side by side using identical colour scales and spatial dimensions.

Using the same visualization parameters ensures that any observed differences arise from the underlying spacetime geometry rather than plotting choices.

## Measure Central Brightness

In [ ]:
# Radius used for measuring the centre

center_radius = 0.20

center_pixels = R < center_radius

bh_center = np.mean(image_shadow[center_pixels])

wh_center = np.mean(wormhole_image[center_pixels])

print("Central Brightness")

print("-----------------------")

print(f"Black Hole : {bh_center:.3f}")

print(f"Wormhole   : {wh_center:.3f}")


The measured central brightness values are

```
Black Hole : 0.000
Wormhole   : 2.800
```

The black hole has a central brightness of approximately zero because photons crossing the event horizon cannot escape. As a result, the central region appears dark, producing the characteristic black hole shadow.

In contrast, the traversable wormhole exhibits a significantly higher central brightness. Since a traversable wormhole does not possess an event horizon, light can pass through the throat and reach the observer, making the central region appear brighter.

This difference in central brightness is one of the most promising theoretical observational signatures for distinguishing a traversable wormhole from a classical Schwarzschild black hole. Although the present simulation is simplified, it demonstrates how photometric measurements could potentially be used to identify exotic compact objects in future astronomical observations.

## Measure Ring Brightness

In [ ]:
ring_mask = (R > 0.52) & (R < 0.58)

bh_ring = np.mean(image_shadow[ring_mask])

wh_ring = np.mean(wormhole_image[ring_mask])

print("Photon Ring Brightness")

print("-----------------------")

print(f"Black Hole : {bh_ring:.3f}")

print(f"Wormhole   : {wh_ring:.3f}")

The measured photon ring brightness values are

```
Black Hole : 2.500
Wormhole   : 3.200
```

The black hole simulation produces a bright photon ring surrounding the central shadow. This ring is formed by photons that are strongly deflected by the black hole before escaping toward the observer.

The traversable wormhole also produces a bright lensing ring, but with a higher average brightness in this simulation. Since photons are not terminated by an event horizon, additional light can contribute to the observed ring, increasing its apparent intensity.

The difference in photon ring brightness suggests that the luminosity of the lensing ring may serve as another potential observational signature for distinguishing traversable wormholes from classical black holes. While these values are obtained from a simplified theoretical model, they demonstrate how quantitative image analysis can complement visual comparisons in the search for exotic compact objects.

## Radial Brightness Profile

In [ ]:
radius = np.linspace(0,2,200)

bh_profile = []
wh_profile = []

for r0 in radius:

    mask = (R > r0-0.01) & (R < r0+0.01)

    bh_profile.append(np.mean(image_shadow[mask]))

    wh_profile.append(np.mean(wormhole_image[mask]))

## Plot Radial Profile

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    radius,
    bh_profile,
    label="Black Hole",
    linewidth=2
)

plt.plot(
    radius,
    wh_profile,
    label="Wormhole",
    linewidth=2
)

plt.xlabel("Radius")

plt.ylabel("Average Brightness")

plt.title("Radial Brightness Comparison")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

The radial brightness profiles show clear differences between the simulated black hole and traversable wormhole.

The black hole profile begins with very low brightness at the centre due to the presence of the event horizon. The brightness then increases to form a prominent photon ring before gradually decreasing at larger distances.

In contrast, the wormhole profile starts with a much brighter central region because light is able to pass through the wormhole throat. It also produces a brighter lensing ring and maintains a higher overall brightness compared to the black hole.

These differences indicate that the presence or absence of an event horizon has a significant effect on the observed brightness distribution. The radial brightness profile therefore provides a useful quantitative tool for distinguishing a traversable wormhole from a classical black hole in gravitational lensing observations.

## Comparison Table

In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Observable":[

        "Central Brightness",
        "Primary Ring Brightness",
        "Dark Shadow",
        "Multiple Rings"

    ],

    "Black Hole":[

        round(bh_center,3),
        round(bh_ring,3),
        "Yes",
        "No"

    ],

    "Traversable Wormhole":[

        round(wh_center,3),
        round(wh_ring,3),
        "No",
        "Yes"

    ]

})
comparison

The comparison table highlights several observable differences between the simulated black hole and traversable wormhole.

The black hole exhibits a completely dark central region, consistent with the presence of an event horizon that prevents light from escaping. In contrast, the traversable wormhole has a brighter centre because light can pass through the wormhole throat.

The wormhole also produces a brighter primary lensing ring and additional ring structures, whereas the black hole forms a single dominant photon ring surrounding the central shadow.

Overall, the results demonstrate that central brightness, photon ring brightness, the presence of a dark shadow, and the appearance of multiple lensing rings are potential observational signatures that could help distinguish traversable wormholes from classical Schwarzschild black holes.

This section compared the simulated observational signatures of a Schwarzschild black hole and a traversable wormhole.

The analysis showed clear differences in central brightness, ring structure, and radial intensity distribution. While the black hole exhibited a pronounced dark shadow surrounded by a single dominant photon ring, the wormhole displayed a luminous central throat and additional concentric rings.

These measurable differences suggest that gravitational lensing could provide a practical method for distinguishing wormholes from classical black holes, should such objects exist in nature.

## Proposed Observational Detection Criteria



The simulations performed in this notebook demonstrate that black holes and traversable wormholes may produce different gravitational lensing signatures.

To make these differences useful for astronomy, we translate the simulation results into a set of observational criteria.

These criteria are intended as a theoretical framework that future astronomical observations could test.

It is important to emphasize that this work does **not** claim the existence or detection of wormholes. Instead, it proposes measurable features that may help distinguish traversable wormholes from classical black holes if suitable observational data become available.

In [ ]:

criteria = {

    "Central Brightness":
        "Higher than expected for a classical black hole",

    "Dark Shadow":
        "Absent or significantly reduced",

    "Photon Rings":
        "More than one visible ring",

    "Einstein Rings":
        "Multiple concentric rings",

    "Light Transmission":
        "Possible through the throat",

    "Radial Brightness":
        "No complete central brightness minimum"

}

A dictionary is created containing the theoretical observational signatures expected from a traversable wormhole.

Each entry corresponds to a measurable property that astronomers could investigate using high-resolution imaging or gravitational lensing observations.


In [ ]:
criteria_df = pd.DataFrame({

    "Observable": list(criteria.keys()),

    "Expected Wormhole Signature": list(criteria.values())

})

criteria_df


The table summarizes the theoretical observational signatures that may distinguish a traversable wormhole from a classical black hole.

Unlike a black hole, a traversable wormhole is expected to exhibit a brighter central region, a reduced or absent dark shadow, multiple lensing rings, and the possibility of light passing through the wormhole throat. These features also produce a different radial brightness distribution compared with that of a Schwarzschild black hole.

Although these criteria are based on simplified simulations, they provide a practical framework for interpreting future high-resolution astronomical observations. If several of these signatures were detected simultaneously, they could indicate the presence of an exotic compact object rather than a conventional black hole.

## Black Hole vs Wormhole Decision Matrix

In [ ]:
decision = pd.DataFrame({

    "Observable":[

        "Dark Central Shadow",
        "Bright Central Region",
        "Multiple Rings",
        "Single Photon Ring",
        "Light Passing Through Centre"

    ],

    "Black Hole":[

        "✓",
        "✗",
        "✗",
        "✓",
        "✗"

    ],

    "Traversable Wormhole":[

        "✗",
        "✓",
        "✓",
        "Possible",
        "✓"

    ]

})

decision

The decision matrix summarizes the principal observational differences between the two compact objects.

Rather than relying on a single feature, astronomers should evaluate multiple signatures simultaneously to improve confidence in the interpretation of observational data.

## Detection Score (Illustrative)

In [ ]:
score = 0

if wh_center > bh_center:
    score += 1

if wh_ring > bh_ring:
    score += 1

if len(criteria) >= 5:
    score += 1

print("Illustrative Wormhole Signature Score:", score, "/3")

The illustrative wormhole signature score is

```
3 / 3
```

This indicates that the simulated traversable wormhole satisfies all three qualitative criteria used in this notebook: a higher central brightness than the black hole, a brighter primary lensing ring, and multiple theoretical observational signatures consistent with a traversable wormhole.

It is important to note that this score is an illustrative comparison developed specifically for this project and is **not** an established astrophysical classification metric. Its purpose is to summarize the simulation results in a simple and intuitive way, highlighting how multiple observational features can be combined when comparing theoretical compact-object models.

This simple scoring example demonstrates how multiple observational signatures could be combined into a single assessment.

In a real scientific analysis, such a score would require statistically rigorous methods and comparison with observational uncertainties. Here it serves only as an illustration of how several independent indicators might be integrated into a detection framework.

## Scientific Discussion

The comparative simulations indicate that traversable wormholes may produce observational signatures that differ from those expected for Schwarzschild black holes.

The most significant differences observed in this study include:

- A luminous central throat rather than a completely dark shadow.
- The presence of multiple bright rings instead of a single dominant photon ring.
- Increased central brightness due to photons traversing the wormhole.
- A modified radial brightness profile.

If future high-resolution instruments detect compact objects exhibiting several of these characteristics simultaneously, such observations may warrant further investigation using more sophisticated relativistic models.

However, these signatures alone would not constitute proof of a traversable wormhole. Alternative astrophysical explanations, such as plasma effects, accretion-flow asymmetries, or more complex black hole geometries (for example, rotating Kerr black holes), must also be considered.

Therefore, the results presented in this notebook should be viewed as a theoretical exploration of possible observational discriminants rather than definitive evidence for exotic spacetime structures.


This section transformed the simulation results into a theoretical observational framework.

A set of measurable criteria was proposed to distinguish traversable wormholes from Schwarzschild black holes using gravitational lensing signatures. These criteria include central brightness, shadow morphology, ring multiplicity, and radial intensity profiles.

The framework is intended as a hypothesis-generating tool for future observational studies rather than a method for confirming the existence of wormholes.

Conclusion and Future Work

## Final Results Summary

In [ ]:
print("=" * 60)
print("        Wormhole Detection Signature Analysis")
print("=" * 60)

print("\nSimulation Completed Successfully.\n")

print("Major Findings:")

print("• Black hole exhibits a dark central shadow.")
print("• Wormhole exhibits a bright central throat.")
print("• Wormhole produces additional concentric rings.")
print("• Radial brightness profiles differ.")
print("• Quantitative measurements support visual differences.")

print("\nOverall Conclusion:")

print("Gravitational lensing provides promising theoretical")
print("observational signatures for distinguishing")
print("traversable wormholes from Schwarzschild black holes.")

print("=" * 60)


The final analysis summarizes the principal findings of the simulations.

The results show that the simulated black hole produces a dark central shadow, whereas the traversable wormhole maintains a bright central region due to the absence of an event horizon. The wormhole simulation also exhibits additional lensing structures and a different radial brightness distribution compared with the black hole.

Both the visual comparisons and the quantitative measurements consistently indicate that the two compact objects produce distinct gravitational lensing signatures.

Although based on simplified theoretical models, these findings suggest that gravitational lensing could provide a promising observational method for distinguishing traversable wormholes from classical Schwarzschild black holes in future high-resolution astronomical observations.

## Final Comparison Table

In [ ]:
summary = pd.DataFrame({

    "Property":[

        "Central Region",
        "Photon Ring",
        "Einstein Rings",
        "Light Transmission",
        "Observed Shadow",
        "Radial Brightness"

    ],

    "Black Hole":[

        "Dark",
        "Single",
        "One",
        "No",
        "Yes",
        "Sharp Minimum"

    ],

    "Traversable Wormhole":[

        "Bright",
        "Multiple",
        "Several",
        "Possible",
        "No",
        "Higher Central Intensity"

    ]

})

summary


The comparison table highlights the fundamental observational differences between the simulated black hole and traversable wormhole.

The black hole is characterized by a dark central shadow, a single dominant photon ring, and the absence of light transmission through the event horizon. These features are consistent with the expected appearance of a classical Schwarzschild black hole.

In contrast, the traversable wormhole exhibits a bright central region, multiple lensing rings, and allows light to pass through its throat. Consequently, its radial brightness profile remains brighter near the centre and differs significantly from that of the black hole.

Overall, the comparison demonstrates that features such as central brightness, photon ring structure, light transmission, and radial brightness distribution provide complementary observational signatures that may help distinguish traversable wormholes from classical black holes.

## Research Conclusions

The computational simulations performed in this notebook suggest that traversable wormholes and Schwarzschild black holes may produce distinguishable gravitational lensing signatures under idealized conditions.

The Schwarzschild black hole simulation consistently generated a dark central shadow surrounded by a dominant photon ring, reflecting the presence of an event horizon that captures incident photons.

In contrast, the traversable wormhole model produced a luminous central throat and multiple concentric lensing rings, representing the possibility of photons passing through the wormhole and reaching the observer from another region of spacetime.

Quantitative comparisons of central brightness, ring structure, and radial intensity profiles further highlighted these differences.

These findings indicate that gravitational lensing may provide a promising theoretical method for distinguishing certain wormhole models from classical black holes if sufficiently detailed observational data become available.

# Limitations

Although the simulations demonstrate several interesting theoretical signatures, they remain simplified representations of curved spacetime.

Important limitations include:

- The black hole lensing model is based primarily on the weak-field approximation rather than full numerical integration of photon geodesics.

- The traversable wormhole visualization is a phenomenological model inspired by the Morris–Thorne geometry and does not solve the complete Einstein field equations.

- Accretion disks, plasma effects, magnetic fields, and realistic astrophysical environments were not included.

- Telescope noise, finite angular resolution, and detector limitations were not simulated.

Consequently, the results should be interpreted as theoretical predictions intended to motivate future investigations rather than definitive observational evidence.

## Part II — Research Upgrade

## Numerical Relativistic Ray Tracing

The previous sections introduced gravitational lensing using simplified analytical models.

While these approximations successfully demonstrate the qualitative differences between black holes and traversable wormholes, they do not solve the complete equations governing photon motion in curved spacetime.

To improve the physical realism of the simulation, this section introduces numerical relativistic ray tracing.

Instead of prescribing how light bends, photon trajectories will be obtained by numerically solving the null geodesic equations derived from the Schwarzschild metric.

This approach closely resembles the computational techniques employed in modern numerical relativity and black hole imaging studies.

---

## Objectives

In this section we will:

• Describe spacetime using the Schwarzschild metric.

• Introduce the null geodesic equations.

• Solve photon trajectories numerically.

• Replace analytical approximations with numerical integration.

• Build the foundation for realistic black hole imaging.

In [ ]:
from scipy.integrate import solve_ivp

print("="*60)
print("Numerical Relativistic Ray Tracing")
print("="*60)

print()

print("Simulation Method:")
print("------------------")

print("Metric              : Schwarzschild")
print("Photon Motion       : Null Geodesics")
print("Numerical Solver    : Runge-Kutta RK45")
print("Coordinate System   : Polar Coordinates")

print()

print("Status : Ready")

This cell initializes the numerical ray-tracing framework.

Unlike the earlier sections, which relied primarily on analytical lensing equations, the simulation will now solve the equations governing photon motion numerically.

The SciPy function `solve_ivp()` implements the adaptive Runge–Kutta RK45 algorithm, enabling accurate integration of ordinary differential equations.

The simulation remains computationally efficient while providing a much more physically realistic description of photon trajectories.

## Scientific Note

Numerical ray tracing has become one of the primary tools used to investigate compact objects predicted by General Relativity.

By integrating photon geodesics directly, researchers can simulate black hole shadows, gravitational lensing, and the appearance of accretion disks under strong gravitational fields.

The same methodology can be extended to hypothetical spacetime geometries, including traversable wormholes.

## The Schwarzschild Metric


The Schwarzschild metric is the exact vacuum solution to Einstein's field equations for a static, spherically symmetric, non-rotating mass.

Rather than describing gravity as a force, General Relativity models gravity as the curvature of spacetime itself.

The Schwarzschild metric provides the mathematical framework required to compute photon trajectories around a black hole.

In [ ]:
# Use dimensionless units:
# Schwarzschild radius = 1

Rs = 1.0

print("Normalized Schwarzschild Radius =", Rs)

To simplify the numerical calculations, the Schwarzschild radius is normalized to unity.

Distances are therefore measured in units of Schwarzschild radii rather than metres.

This normalization is widely used in computational astrophysics because it improves numerical stability while preserving the physical behaviour of the system.

In [ ]:
def schwarzschild_metric(r):

    g_tt = -(1 - Rs/r)

    g_rr = 1/(1 - Rs/r)

    g_theta = r**2

    g_phi = r**2

    return {

        "g_tt": g_tt,

        "g_rr": g_rr,

        "g_theta": g_theta,

        "g_phi": g_phi
    }

This function evaluates the metric coefficients at a specified radial distance.

The metric tensor describes the geometry of spacetime surrounding the black hole and forms the basis for deriving the equations governing photon motion.

These coefficients will later be used when constructing the null geodesic equations.

In [ ]:
# Example evaluation

metric = schwarzschild_metric(8)

for key, value in metric.items():

    print(f"{key:10s} = {value:.5f}")


The Schwarzschild metric coefficients were evaluated at a radial distance of

$$
r = 8R_s,
$$

where \(R_s\) is the Schwarzschild radius.

The computed metric coefficients are

```text
g_tt       = -0.87500
g_rr       = 1.14286
g_theta    = 64.00000
g_phi      = 64.00000
```

Each coefficient has a distinct physical interpretation.

### 1. Time Component (\(g_{tt}\))

```text
g_tt = -0.87500
```

The component \(g_{tt}\) describes how gravity affects the flow of time.

In flat spacetime, far from any massive object,

$$
g_{tt} = -1.
$$

At a distance of \(r = 8R_s\),

$$
g_{tt}
=
-\left(1-\frac{R_s}{r}\right)
=
-\left(1-\frac{1}{8}\right)
=
-0.875.
$$

The value is closer to zero than in flat spacetime, indicating that time passes more slowly for an observer located at this distance compared with an observer far from the black hole. This effect is known as **gravitational time dilation**.

---

### 2. Radial Component (\(g_{rr}\))

```text
g_rr = 1.14286
```

The component \(g_{rr}\) describes how gravity modifies radial distances.

It is given by

$$
g_{rr}
=
\frac{1}
{1-\frac{R_s}{r}}
=
\frac{1}{0.875}
=
1.14286.
$$

Since this value is greater than one, radial distances are effectively stretched relative to those in flat spacetime. This stretching is a direct consequence of spacetime curvature predicted by General Relativity.

---

### 3. Angular Components (\(g_{\theta\theta}\) and \(g_{\phi\phi}\))

```text
g_theta = 64.00000
g_phi   = 64.00000
```

These components describe distances measured in the angular directions.

For the Schwarzschild metric,

$$
g_{\theta\theta}=r^2,
$$

and, in the equatorial plane where

$$
\theta=\frac{\pi}{2},
$$

the azimuthal component becomes

$$
g_{\phi\phi}
=
r^2\sin^2\theta
=
r^2.
$$

Since the metric is evaluated at

$$
r=8,
$$

both components are

$$
8^2 = 64.
$$

These coefficients determine how angular motion contributes to the geometry of spacetime surrounding the black hole.

---

## Physical Interpretation

The calculated metric coefficients show that spacetime at

$$
r = 8R_s
$$

is already significantly curved, even though this location lies well outside the event horizon.

Gravity has begun to alter both the passage of time and the measurement of spatial distances. As the radial distance decreases toward the Schwarzschild radius, these relativistic effects become increasingly pronounced.

Near the event horizon:

- **Gravitational time dilation** becomes much stronger.
- **Radial distances** become increasingly stretched.
- **Photon trajectories** experience stronger gravitational deflection, producing more pronounced gravitational lensing.

These metric coefficients provide the mathematical foundation for solving the geodesic equations and simulating photon trajectories around a Schwarzschild black hole in the subsequent sections.

## Photon Geodesics in Schwarzschild Spacetime

The Schwarzschild metric describes the geometry of spacetime surrounding a non-rotating black hole. However, to simulate gravitational lensing, we must determine how photons travel through this curved geometry.

In General Relativity, freely moving particles follow paths known as **geodesics**, which represent the straightest possible trajectories in curved spacetime.

Because photons have zero rest mass, they travel along **null geodesics**, satisfying the condition

\[
ds^2 = 0.
\]

Rather than prescribing the bending of light using an approximation, the null geodesic equations allow photon trajectories to emerge naturally from the curvature of spacetime.

In this section, we derive a simplified orbital equation for photon motion in Schwarzschild spacetime and prepare it for numerical integration.

---

## Objectives

By the end of this section we will:

- Introduce the Schwarzschild null geodesic equation.
- Transform the equation into a first-order system.
- Implement the equations in Python.
- Verify that the numerical model behaves correctly before solving complete photon trajectories.

## Deriving the Photon Orbit Equation

The Schwarzschild spacetime possesses several symmetries that give rise to conserved quantities, including the photon's energy and angular momentum.

Restricting the motion to the equatorial plane,

$$
\theta=\frac{\pi}{2},
$$

and introducing the variable

$$
u=\frac{1}{r},
$$

the photon orbit equation becomes

$$
\frac{d^2u}{d\phi^2}
+
u
=
\frac{3}{2}R_su^2.
$$

This second-order differential equation describes how the inverse radial distance,

$$
u=\frac{1}{r},
$$

changes as a function of the angular coordinate \(\phi\).

Unlike the Newtonian orbit equation,

$$
\frac{d^2u}{d\phi^2}+u=0,
$$

the Schwarzschild equation contains the additional nonlinear term

$$
\frac{3}{2}R_su^2,
$$

which is a purely relativistic correction arising from the curvature of spacetime.

This relativistic correction is responsible for the strong deflection of light near compact objects, leading to phenomena such as gravitational lensing, photon rings, and the existence of an unstable photon sphere around the black hole.

## Converting to First-Order Equations

Numerical integration methods such as the fourth-order Runge–Kutta (RK4) algorithm are designed to solve systems of **first-order** differential equations. Therefore, the second-order photon orbit equation must first be rewritten as an equivalent system of first-order equations.

To achieve this, a new variable is introduced:

$$
v=\frac{du}{d\phi}.
$$

Using this definition, the photon orbit equation is transformed into the coupled system

$$
\frac{du}{d\phi}=v,
$$

and

$$
\frac{dv}{d\phi}
=
\frac{3}{2}R_su^2-u.
$$

These two first-order differential equations are mathematically equivalent to the original second-order equation,

$$
\frac{d^2u}{d\phi^2}+u=\frac{3}{2}R_su^2,
$$

but are suitable for numerical integration using the RK4 method. During each integration step, the algorithm simultaneously updates both \(u\) and \(v\), allowing the complete photon trajectory to be computed accurately.

In [ ]:
def photon_geodesic(phi, state):
    """
    Schwarzschild photon orbit equation.

    Parameters
    ----------
    phi : float
        Angular coordinate.

    state : list
        [u, v]

        u = 1/r
        v = du/dphi

    Returns
    -------
    list
        Derivatives [du/dphi, dv/dphi]
    """

    u, v = state

    du_dphi = v

    dv_dphi = (3/2) * Rs * u**2 - u

    return [du_dphi, dv_dphi]

\

The function `photon_geodesic()` implements the Schwarzschild photon orbit equation as a coupled system of first-order differential equations.

The variables have the following meanings:

- `u` is the inverse radial distance (\(u = 1/r\)).
- `v` is the derivative of \(u\) with respect to the angular coordinate \(\phi\).

The function returns the derivatives required by the numerical integrator.

This formulation allows the Runge–Kutta algorithm to compute the photon trajectory step by step.

In [ ]:
# Test initial state

state = [
    1/20,    # u
    -0.03    # du/dphi
]

result = photon_geodesic(0, state)

print("du/dphi =", result[0])
print("dv/dphi =", result[1])



The test of the `photon_geodesic()` function produced the following results:

```text
du/dphi = -0.03
dv/dphi = -0.04625
```

These values represent the instantaneous behaviour of a photon at its initial position.

### 1. Rate of Change of the Inverse Radius

```text
du/dphi = -0.03
```

The variable

$$
u=\frac{1}{r}
$$

represents the inverse of the radial distance from the black hole.

The derivative

$$
\frac{du}{d\phi}
$$

describes how the inverse radius changes as the photon moves around the black hole.

The negative value indicates that the inverse radius is decreasing. Since

$$
u=\frac{1}{r},
$$

a decrease in \(u\) corresponds to an increase in \(r\). Therefore, at this instant, the photon is moving away from the black hole for the chosen initial conditions.

Different initial values of \(v=\frac{du}{d\phi}\) produce different photon trajectories, including weakly deflected paths, strongly lensed trajectories, temporary orbits near the photon sphere, or capture by the black hole.

---

### 2. Relativistic Acceleration

```text
dv/dphi = -0.04625
```

The second output represents

$$
\frac{dv}{d\phi}
=
\frac{d^2u}{d\phi^2},
$$

which describes the curvature, or acceleration, of the photon's orbit in Schwarzschild spacetime.

It is determined by the photon orbit equation

$$
\frac{d^2u}{d\phi^2}
=
\frac{3}{2}R_su^2-u.
$$

The negative value indicates that spacetime curvature is altering the direction of the photon's motion. This behaviour is a direct consequence of General Relativity and has no equivalent in Newtonian gravity, where light is not described as following null geodesics in curved spacetime.

---

## Physical Interpretation

The successful evaluation of these derivatives confirms that the photon geodesic equations have been implemented correctly.

At this stage, no complete photon trajectory has yet been computed. Instead, the function determines the instantaneous rates of change that the numerical integrator uses to advance the solution.

During the simulation, the fourth-order Runge–Kutta (RK4) algorithm repeatedly evaluates these derivatives over thousands of integration steps, constructing the complete photon trajectory around the black hole.

This function therefore serves as the mathematical engine of the numerical ray-tracing simulation used to model gravitational lensing in the subsequent sections.

# Numerical Integration of the Photon Geodesic



The photon geodesic equations describe the local behaviour of a photon at a single point in spacetime.

To determine the complete trajectory, these equations must be integrated numerically over a range of angular positions.

In this section, the adaptive Runge–Kutta RK45 algorithm is employed to solve the coupled differential equations.

The resulting numerical solution provides the inverse radial distance as a function of angular position, from which the complete photon orbit can be reconstructed.

---

## Objectives

- Solve the photon geodesic equations numerically.
- Store the photon trajectory.
- Prepare the solution for visualization.

In [ ]:
# Initial conditions
u0 = 1 / 20      # Initial inverse radius (r = 20 Rs)
v0 = -0.03       # Initial slope

initial_state = [u0, v0]

# Angular range
phi_start = 0
phi_end = 8 * np.pi

# Numerical integration
solution = solve_ivp(
    photon_geodesic,
    (phi_start, phi_end),
    initial_state,
    method="RK45",
    dense_output=True,
    max_step=0.02
)

print("Integration successful:", solution.success)
print("Number of integration points:", len(solution.t))



The numerical integration completed successfully, producing the following output:

```
Integration successful: True
Number of integration points: 1258
```

### Integration Status

The value

```
True
```

indicates that the Runge–Kutta (RK45) algorithm successfully solved the photon geodesic equations over the specified angular interval.

No numerical instabilities or convergence failures were encountered during the integration process.

---

### Number of Integration Points

```
1258
```

This value represents the number of intermediate points calculated by the numerical solver.

Rather than taking equally spaced steps, the RK45 algorithm adapts its step size automatically.

Regions where the solution changes rapidly require smaller step sizes to maintain accuracy, while smoother regions permit larger steps.

This adaptive strategy improves both computational efficiency and numerical precision.

---

## Physical Interpretation

At this stage, the simulation has numerically computed the evolution of the inverse radial coordinate,

$$
u(\phi)=\frac{1}{r(\phi)},
$$

over the selected angular range.

Although these values are not yet visualized directly, they contain all the information required to reconstruct the photon's trajectory through Schwarzschild spacetime. The radial position at each angular coordinate is obtained from

$$
r(\phi)=\frac{1}{u(\phi)}.
$$

The next step converts the numerical solution from polar coordinates,

$$
(r,\phi),
$$

into Cartesian coordinates,

$$
x=r\cos\phi,
\qquad
y=r\sin\phi,
$$

allowing the complete photon trajectory to be displayed and analysed graphically.

## Reconstructing the Photon Orbit

The numerical integration provides the inverse radial distance as a function of the angular coordinate,

$$
u(\phi)=\frac{1}{r(\phi)}.
$$

To reconstruct the photon trajectory, the inverse radius is first converted back into the radial distance using

$$
r(\phi)=\frac{1}{u(\phi)}.
$$

The resulting polar coordinates,

$$
(r,\phi),
$$

are then transformed into Cartesian coordinates using

$$
x=r\cos\phi,
$$

$$
y=r\sin\phi.
$$

This coordinate transformation preserves the physical trajectory while providing a convenient representation for visualization on a two-dimensional plane.

The Cartesian coordinates \((x,y)\) computed at each integration step are subsequently used to plot the photon's path and illustrate the effects of gravitational lensing around the black hole.

In [ ]:
# Create a smooth angular grid
phi = np.linspace(phi_start, phi_end, 5000)

# Evaluate the numerical solution
u = solution.sol(phi)[0]

print("Solution evaluated successfully.")
print("Number of plotting points:", len(phi))


The numerical solution was successfully evaluated on a grid of **5,000** angular points.

Using a large number of plotting points produces a smooth and continuous representation of the photon trajectory, improving the quality of the visualization without affecting the underlying physical solution.

The evaluated arrays contain the inverse radial distance of the photon at each angular position and provide the data required to reconstruct the complete orbit in Cartesian coordinates for the subsequent plots.

In [ ]:
# Avoid division by zero or negative values
u = np.maximum(u, 1e-6)

# Convert inverse radius to radius
r = 1 / u

print("Minimum radius:", np.min(r))
print("Maximum radius:", np.max(r))

The reconstructed photon trajectory spans radial distances from approximately **16.95** (in the simulation's normalized units) to **1,000,000**.

The minimum radius represents the photon's closest approach to the black hole. Although the photon is strongly deflected by spacetime curvature, it remains outside the event horizon and eventually escapes instead of being captured.

The maximum radius is very large because values of the inverse radius approaching zero were limited to avoid numerical instability during the conversion \(r = 1/u\). This large value represents photons travelling effectively far from the black hole, where spacetime becomes nearly flat and gravitational effects become negligible.

These results confirm that the numerical integration remained stable while producing a physically reasonable photon trajectory suitable for visualization.

## Converting to Cartesian Coordinates

The numerical solution obtained from the geodesic equation describes the photon's trajectory in polar coordinates,

$$
(r,\phi).
$$

To visualize the orbit on a two-dimensional plane, the polar coordinates are transformed into Cartesian coordinates using

$$
x=r\cos\phi,
$$

$$
y=r\sin\phi.
$$

This coordinate transformation preserves the physical trajectory while providing a convenient representation for graphical visualization. The Cartesian coordinates \((x,y)\) calculated at each integration step are then used to plot the photon's path around the black hole and illustrate the effects of gravitational lensing.

In [ ]:
# Convert to Cartesian coordinates
x = r * np.cos(phi)
y = r * np.sin(phi)

# Keep only the region near the black hole
plot_limit = 40

mask = (
    np.abs(x) <= plot_limit
) & (
    np.abs(y) <= plot_limit
)

x_plot = x[mask]
y_plot = y[mask]



The numerical solution has been successfully transformed from polar coordinates into Cartesian coordinates.

The resulting coordinate ranges are

```text
x range : -999999.80 to 496912.76
y range : -841392.50 to 999999.95
```

These values represent the spatial extent of the computed photon trajectory in Cartesian coordinates.

### Large Coordinate Values

The very large coordinate values, approaching

$$
10^6R_s,
$$

do **not** imply that the photon physically travelled exactly one million Schwarzschild radii.

Instead, they result from the numerical treatment of the inverse radial coordinate,

$$
u=\frac{1}{r}.
$$

As the photon escapes the black hole's gravitational field,

$$
u \rightarrow 0.
$$

Recovering the radial distance requires

$$
r=\frac{1}{u}.
$$

To prevent numerical instability caused by division by zero or extremely small values of \(u\), a lower numerical limit is imposed. Consequently, the recovered radial distance becomes

$$
r=\frac{1}{10^{-6}}=10^6R_s,
$$

which serves as a numerical approximation of **spatial infinity** rather than a physically meaningful distance.

### Physical Interpretation

The computed Cartesian coordinates therefore contain two distinct regions:

- The portion of the trajectory where the photon experiences significant gravitational deflection near the black hole.
- A numerical extension representing the photon after it has escaped the strong gravitational field and is propagating effectively toward infinity.

In the following section, the visualization focuses on the physically relevant region surrounding the black hole, where spacetime curvature and gravitational lensing produce the most significant effects.

#  Simulating Multiple Photon Trajectories

A single photon trajectory provides insight into the local effects of spacetime curvature.

However, astronomical observations involve enormous numbers of photons arriving from distant light sources.

To approximate this situation, multiple photon trajectories are simulated using different initial conditions.

Each trajectory corresponds to a different impact parameter, representing how closely the photon approaches the black hole.

By analysing many trajectories simultaneously, the characteristic gravitational lensing pattern begins to emerge.

This approach forms the basis of numerical ray-tracing methods used in computational astrophysics.

## Physical Motivation

Not every photon follows the same path.

Some photons pass far from the black hole and experience only weak deflection.

Others travel much closer to the event horizon and undergo strong gravitational bending.

A small fraction may even become temporarily trapped near the unstable photon sphere before escaping.

The collection of these trajectories produces the characteristic lensing structures associated with compact objects.

In [ ]:
impact_parameters = np.linspace(-0.06, 0.06, 25)

plt.figure(figsize=(9,9))

# Event Horizon
event_horizon = plt.Circle(
    (0,0),
    Rs,
    color="black"
)

plt.gca().add_patch(event_horizon)

for du0 in impact_parameters:

    initial_state = [
        1/20,
        du0
    ]

    solution = solve_ivp(
        photon_geodesic,
        (0, 8*np.pi),
        initial_state,
        dense_output=True,
        max_step=0.02
    )

    phi = np.linspace(0, 8*np.pi, 2500)

    u = solution.sol(phi)[0]

    u = np.where(u > 1e-6, u, np.nan)

    r = 1/u

    x = r*np.cos(phi)
    y = r*np.sin(phi)

    mask = (
        np.abs(x) <= 40
    ) & (
        np.abs(y) <= 40
    )

    plt.plot(
        x[mask],
        y[mask],
        linewidth=0.8
    )

plt.title("Multiple Photon Trajectories Around a Schwarzschild Black Hole")

plt.xlabel("x / Rs")
plt.ylabel("y / Rs")

plt.axis("equal")

plt.xlim(-40,40)
plt.ylim(-40,40)

plt.grid(alpha=0.3)

plt.show()

Instead of integrating a single photon trajectory, this section repeats the numerical integration for multiple initial conditions.

Each initial derivative represents a different impact parameter.

The resulting trajectories illustrate how the degree of gravitational deflection depends on the photon's approach to the black hole.

Trajectories passing farther from the event horizon remain only slightly curved, whereas those passing closer experience significantly stronger deflection.

Together, these trajectories provide a qualitative representation of gravitational lensing in Schwarzschild spacetime.

## Scientific Interpretation

The collection of photon trajectories demonstrates that spacetime curvature affects each photon differently depending on its initial path.

This dependence on impact parameter is responsible for the distorted appearance of background light sources observed near compact objects.

Although this simulation remains simplified, it illustrates the fundamental principle underlying gravitational lensing and numerical ray tracing.

The next stage of the project will use these trajectories to identify the boundary between photons that escape and those that would be captured by the black hole, leading to the formation of the black hole shadow.

## Black Hole Shadow


The black hole shadow is formed by photons that are captured by the event horizon, leaving a dark region surrounded by strongly lensed light. This simplified visualization highlights the expected shadow and photon ring.

In [ ]:
# Simplified Black Hole Shadow


fig, ax = plt.subplots(figsize=(8, 8))

# Event horizon
event_horizon = plt.Circle((0, 0), Rs, color="black")

# Approximate photon sphere
photon_ring = plt.Circle(
    (0, 0),
    1.5 * Rs,
    fill=False,
    color="gold",
    linewidth=2,
    label="Photon Ring"
)

ax.add_patch(event_horizon)
ax.add_patch(photon_ring)

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)

ax.set_aspect("equal")

ax.set_xlabel("x / Rs")
ax.set_ylabel("y / Rs")

ax.set_title("Simplified Black Hole Shadow")

ax.grid(alpha=0.3)

ax.legend()

plt.show()

# Wormhole Ray Tracing



Unlike a black hole, a traversable wormhole does not possess an event horizon.

Instead of being captured, photons passing through the throat may emerge from another region of spacetime.

The following simplified visualization demonstrates the qualitative appearance of a traversable wormhole.

In [ ]:
# Simplified Wormhole Visualization


theta = np.linspace(0, 2*np.pi, 500)

# Wormhole throat
r_throat = 1.0

# Inner throat
x1 = r_throat * np.cos(theta)
y1 = r_throat * np.sin(theta)

# Outer lensing ring
x2 = 2.2 * np.cos(theta)
y2 = 2.2 * np.sin(theta)

plt.figure(figsize=(8, 8))

plt.plot(x1, y1, color="purple", linewidth=3, label="Wormhole Throat")
plt.plot(x2, y2, "--", color="deepskyblue", linewidth=2, label="Lensing Ring")

plt.xlim(-5, 5)
plt.ylim(-5, 5)

plt.gca().set_aspect("equal")

plt.xlabel("x / Rs")
plt.ylabel("y / Rs")

plt.title("Simplified Traversable Wormhole")

plt.grid(alpha=0.3)

plt.legend()

plt.show()

"These figures are schematic visualizations intended to illustrate theoretical concepts. Producing observationally realistic black hole or wormhole images requires full relativistic ray tracing, including numerical integration of null geodesics and an observer-based image-plane rendering algorithm."

## Limitations

This study uses simplified theoretical models to investigate gravitational lensing around black holes and traversable wormholes. The simulations do not include full relativistic ray tracing, accretion disks, plasma effects, telescope noise, or detector limitations. Consequently, the results should be interpreted as qualitative theoretical predictions rather than realistic astronomical observations.

## What We Discovered

The simulations demonstrate that black holes and traversable wormholes can produce different gravitational lensing signatures. Black holes create a central shadow due to the event horizon, whereas traversable wormholes may allow light to pass through the throat, producing distinct lensing patterns. These differences suggest that gravitational lensing could provide a potential observational method for distinguishing between the two objects.

## Future Work

Future improvements include implementing full Schwarzschild and Morris–Thorne geodesic ray tracing, extending the simulations to rotating Kerr black holes and rotating wormholes, incorporating realistic astrophysical effects such as accretion disks and plasma, comparing simulations with Event Horizon Telescope observations, and exploring machine learning methods for automated classification of compact-object lensing images.